# ARC-v0.8 — Sealed HotpotQA H1–H4 Cross-Dataset Confirmation

This notebook is the **primary cross-dataset confirmation** of the FEVER approximation-feedback mechanism.

It is sealed **after ARC-v0.7 baseline infrastructure validation but before any HotpotQA iterative-feedback H1–H4 retrieval**.

Primary hypotheses, copied from ARC-v0.5:

- **H1:** mean query-state divergence slope \(>0\)
- **H2:** mean baseline-normalized candidate-divergence slope \(>0\)
- **H3:** mean absolute nDCG@10 utility-gap slope \(>0\)
- **H4:** final-round \(nDCG@10_B-nDCG@10_A>0\), where
  - \(A=\) PQ32 search → PQ32 feedback
  - \(B=\) PQ32 search → SQ8 feedback

Frozen design:

- HotpotQA official DEV: 5,447 queries
- 5,233,329-document corpus
- BAAI/bge-small-en-v1.5
- IVF-PQ32 vs IVF-SQ8
- nlist=4096, nprobe=64
- Top-100 retrieval
- 4 feedback rounds
- FEVER-frozen feedback configs:
  - mean, k=20, alpha=0.3
  - softmax, k=5, alpha=0.5, temperature=0.1
- anchored update to original \(q_0\)
- query as primary independent unit
- average within query across the two frozen feedback configs
- 20,000 bootstrap resamples
- 20,000 one-sided paired sign-flip samples
- Holm correction jointly across H1–H4
- full confirmation passes only if all four endpoints pass
- TEST remains untouched

PQ64 is not part of the primary H1–H4 decision and is reserved for later fidelity dose-response analysis.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
%pip install -q faiss-cpu==1.12.0 psutil pyarrow


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, gc, sqlite3, time

import numpy as np
import pandas as pd
import faiss
import psutil
import matplotlib.pyplot as plt

print("FAISS:", faiss.__version__)
print("NumPy:", np.__version__)
print("System RAM GB:", psutil.virtual_memory().total / 1024**3)


## 1. Frozen constants and artifact paths


In [ ]:
SEED = 20260816
DIM = 384
N_DOCS = 5_233_329
DEV_QUERY_COUNT = 5_447

TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4
NLIST = 4096
NPROBE = 64

BOOTSTRAP_SAMPLES = 20_000
RANDOMIZATION_SAMPLES = 20_000
RESAMPLE_CHUNK = 250

ROOT = Path("/content/drive/MyDrive/hc-rars-external-confirmation-hotpotqa-5m-v1")
SHARD_ROOT = ROOT / "stage1/corpus-embedding-shards-v3"
SHARD_MANIFEST = SHARD_ROOT / "manifest.json"

QUERY_EMB = ROOT / "stage1/query_embeddings.float32.npy"
QUERY_IDS = ROOT / "stage1/query_ids.utf8.txt"
SPLIT_MANIFEST = ROOT / "stage1/official_split_manifest.json"
CORPUS_DB = ROOT / "stage1/corpus_ids.sqlite"
DEV_QRELS = ROOT / "source/hotpotqa/qrels/dev.tsv"

ARC_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-v0")
INDEX_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache/hotpotqa-rebuilt-v3")

PQ32_PATH = INDEX_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss"
PQ64_PATH = INDEX_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfpq-nlist4096-m64-nbits8-seed20260816.faiss"
SQ8_PATH = INDEX_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfsq8-nlist4096-seed20260816.faiss"

CONFIRM_ROOT = ARC_ROOT / "sealed-hotpotqa-h1-h4-confirmation-v08"
CONFIRM_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = CONFIRM_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

for p in [SHARD_MANIFEST, QUERY_EMB, QUERY_IDS, SPLIT_MANIFEST, CORPUS_DB, DEV_QRELS, PQ32_PATH, PQ64_PATH, SQ8_PATH]:
    if not p.is_file():
        raise FileNotFoundError(p)

print("Output:", OUT)


## 2. Frozen feedback configs and provenance


In [ ]:
FROZEN_FEEDBACK_CONFIGS = [
    {"method": "mean", "k": 20, "alpha": 0.3, "temperature": None},
    {"method": "softmax", "k": 5, "alpha": 0.5, "temperature": 0.1},
]

FEVER_RUNS = sorted(
    [p for p in (ARC_ROOT / "sealed-fever-dev-confirmation-v05").glob("*") if (p / "report.json").is_file()],
    key=lambda p: p.stat().st_mtime,
)
if not FEVER_RUNS:
    raise FileNotFoundError("Completed ARC-v0.5 FEVER report not found.")

FEVER_RUN = FEVER_RUNS[-1]
with open(FEVER_RUN / "report.json", "r", encoding="utf-8") as f:
    fever_report = json.load(f)

assert fever_report["feedback_configs"] == FROZEN_FEEDBACK_CONFIGS

V07_RUNS = sorted(
    [p for p in (ARC_ROOT / "hotpotqa-fidelity-index-rebuild-v07").glob("*") if (p / "report.json").is_file()],
    key=lambda p: p.stat().st_mtime,
)
if not V07_RUNS:
    raise FileNotFoundError("Completed ARC-v0.7 report not found.")

V07_RUN = V07_RUNS[-1]
V07_REPORT_PATH = V07_RUN / "report.json"
with open(V07_REPORT_PATH, "r", encoding="utf-8") as f:
    v07_report = json.load(f)

assert v07_report["status"] == "HOTPOTQA_FIDELITY_INDEX_REBUILD_V07_COMPLETE"
assert int(v07_report["corpus_rows"]) == N_DOCS
assert int(v07_report["dev_queries"]) == DEV_QUERY_COUNT
assert int(v07_report["nlist"]) == NLIST
assert int(v07_report["nprobe"]) == NPROBE
assert v07_report["test_qrels_accessed"] is False
assert v07_report["test_retrieval_performed"] is False

print("FEVER source:", FEVER_RUN)
print("v0.7 source:", V07_RUN)
print("Frozen configs:", FROZEN_FEEDBACK_CONFIGS)
display(pd.DataFrame(v07_report["full_dev_baseline"]))


## 3. Hash helpers and rebuilt-shard manifest


In [ ]:
def sha256_file(path, chunk_size=64 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

with open(SHARD_MANIFEST, "r", encoding="utf-8") as f:
    shard_manifest = json.load(f)

assert shard_manifest["status"] == "CORPUS_EMBEDDING_SHARDS_COMPLETE"
shards = sorted(shard_manifest["shards"], key=lambda x: int(x["shard_id"]))
assert sum(int(s["rows"]) for s in shards) == N_DOCS
assert int(shard_manifest["dimension"]) == DIM

SHARD_STARTS = np.asarray([int(s["start_row"]) for s in shards], dtype=np.int64)
SHARD_ENDS = np.asarray([int(s["end_row"]) for s in shards], dtype=np.int64)

print("Shard count:", len(shards))
print("Shard manifest SHA:", sha256_file(SHARD_MANIFEST))


## 4. Seal before HotpotQA H1–H4 iterative-feedback retrieval


In [ ]:
protocol = {
    "schema_version": 1,
    "status": "SEALED_BEFORE_HOTPOTQA_ITERATIVE_FEEDBACK_RETRIEVAL",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "HotpotQA",
    "confirmation_split": "DEV",
    "expected_dev_query_count": DEV_QUERY_COUNT,
    "encoder": "BAAI/bge-small-en-v1.5",
    "corpus_rows": N_DOCS,
    "dimension": DIM,
    "source_corpus_shard_manifest": str(SHARD_MANIFEST),
    "source_corpus_shard_manifest_sha256": sha256_file(SHARD_MANIFEST),
    "source_v07_baseline_report": str(V07_REPORT_PATH),
    "source_v07_baseline_report_sha256": sha256_file(V07_REPORT_PATH),
    "source_fever_v05_report": str(FEVER_RUN / "report.json"),
    "source_fever_v05_report_sha256": sha256_file(FEVER_RUN / "report.json"),
    "top_retrieve": TOP_RETRIEVE,
    "top_k": TOP_K,
    "feedback_rounds": MAX_ROUNDS,
    "nlist": NLIST,
    "nprobe": NPROBE,
    "conditions": {
        "low_fidelity": "IVF-PQ32",
        "high_fidelity": "IVF-SQ8",
        "secondary_mid_fidelity_not_primary": "IVF-PQ64",
    },
    "index_artifacts": {
        "PQ32": {"path": str(PQ32_PATH), "sha256": sha256_file(PQ32_PATH)},
        "PQ64": {"path": str(PQ64_PATH), "sha256": sha256_file(PQ64_PATH)},
        "SQ8": {"path": str(SQ8_PATH), "sha256": sha256_file(SQ8_PATH)},
    },
    "feedback_configs": FROZEN_FEEDBACK_CONFIGS,
    "primary_endpoints": [
        "H1_query_divergence_slope",
        "H2_candidate_increment_slope",
        "H3_abs_utility_gap_slope",
        "H4_B_minus_A",
    ],
    "primary_unit": "query",
    "method_pooling": "average within query across the two FEVER-frozen feedback configs",
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "randomization_samples": RANDOMIZATION_SAMPLES,
    "multiple_testing": "Holm jointly across H1-H4",
    "endpoint_pass_rule": "mean>0 and bootstrap_CI_low>0 and Holm_p<0.05",
    "full_confirmation_rule": "all H1-H4 pass",
    "hotpotqa_h1_h4_based_selection_allowed": False,
    "hotpotqa_baseline_was_observed_for_infrastructure_validation": True,
    "test_retrieval_allowed": False,
    "test_qrels_access_allowed": False,
}

canonical = json.dumps(protocol, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
protocol_sha256 = hashlib.sha256(canonical).hexdigest()
protocol["protocol_sha256"] = protocol_sha256

PROTOCOL_PATH = OUT / "sealed_protocol.json"
PROTOCOL_PATH.write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8")

(OUT / "SEALED_BEFORE_H1_H4_RETRIEVAL.txt").write_text(
    "SEALED\n"
    f"protocol_sha256={protocol_sha256}\n"
    f"created_at_utc={protocol['created_at_utc']}\n"
    "hotpotqa_h1_h4_outcomes_observed_before_seal=False\n"
    "test_accessed=False\n",
    encoding="utf-8",
)

print("SEALED:", PROTOCOL_PATH)
print("Protocol SHA-256:", protocol_sha256)


## 5. Official DEV query/qrels alignment — DEV only


In [ ]:
queries = np.load(QUERY_EMB, mmap_mode="r")

with open(QUERY_IDS, "r", encoding="utf-8") as f:
    query_ids = [x.strip() for x in f if x.strip()]

with open(SPLIT_MANIFEST, "r", encoding="utf-8") as f:
    split = json.load(f)

dev_ids = [str(x) for x in split["dev_query_ids"]]

assert len(dev_ids) == DEV_QUERY_COUNT
assert split["test_qrels_relevance_values_accessed"] is False
assert split["test_retrieval_performed"] is False
assert split["test_outcomes_observed"] is False

query_row = {qid: i for i, qid in enumerate(query_ids)}
missing_queries = [q for q in dev_ids if q not in query_row]
if missing_queries:
    raise RuntimeError(f"Missing DEV query embeddings: {len(missing_queries)}")

dev_query_rows = np.asarray([query_row[q] for q in dev_ids], dtype=np.int64)

assert DEV_QRELS.name == "dev.tsv"
assert "test" not in str(DEV_QRELS).lower()

dev = pd.read_csv(DEV_QRELS, sep="\t")
dev["query-id"] = dev["query-id"].astype(str)
dev["corpus-id"] = dev["corpus-id"].astype(str)
assert set(dev["query-id"]) == set(dev_ids)

unique_doc_ids = dev["corpus-id"].drop_duplicates().tolist()

with sqlite3.connect(str(CORPUS_DB)) as con:
    con.execute("CREATE TEMP TABLE requested_ids (doc_id TEXT PRIMARY KEY)")
    con.executemany("INSERT INTO requested_ids(doc_id) VALUES (?)", [(x,) for x in unique_doc_ids])
    mapped = pd.read_sql_query(
        """
        SELECT r.doc_id, d.row_id
        FROM requested_ids r
        LEFT JOIN documents d ON d.doc_id = r.doc_id
        """,
        con,
    )

assert mapped["row_id"].notna().all()
mapped["row_id"] = mapped["row_id"].astype(np.int64)

dev = dev.merge(
    mapped,
    left_on="corpus-id",
    right_on="doc_id",
    how="left",
    validate="many_to_one",
).rename(columns={"row_id": "corpus-row"})

dev["corpus-row"] = dev["corpus-row"].astype(np.int64)

dev_qrels = {}
for qid, g in dev.groupby("query-id"):
    dev_qrels[str(qid)] = set(
        g.loc[g["score"] > 0, "corpus-row"].astype(np.int64).tolist()
    )

assert len(dev_qrels) == DEV_QUERY_COUNT
assert set(map(len, dev_qrels.values())) == {2}

print("DEV queries:", len(dev_ids))
print("DEV qrels:", len(dev))
print("TEST accessed: False")
print("DEV ALIGNMENT — PASS")


## 6. Fast immutable-shard row loader


In [ ]:
def load_rows_from_shards_fast(rows):
    rows = np.asarray(rows, dtype=np.int64)
    original_shape = rows.shape
    flat = rows.reshape(-1)

    if np.any(flat < 0) or np.any(flat >= N_DOCS):
        raise IndexError("Corpus row out of range")

    unique_rows, inverse = np.unique(flat, return_inverse=True)
    shard_ids = np.searchsorted(SHARD_ENDS, unique_rows, side="right")

    if np.any(shard_ids >= len(shards)):
        raise RuntimeError("Shard lookup failed")

    unique_vectors = np.empty((len(unique_rows), DIM), dtype=np.float32)

    for sid in np.unique(shard_ids):
        mask = shard_ids == sid
        start = int(shards[int(sid)]["start_row"])
        local_rows = unique_rows[mask] - start

        arr = np.load(SHARD_ROOT / shards[int(sid)]["file"], mmap_mode="r")
        unique_vectors[mask] = np.asarray(arr[local_rows], dtype=np.float32)

    result = unique_vectors[inverse]
    return result.reshape(*original_shape, DIM)

rng = np.random.default_rng(SEED)
probe = load_rows_from_shards_fast(
    rng.integers(0, N_DOCS, size=1000, dtype=np.int64)
)
probe_norms = np.linalg.norm(probe, axis=1)

assert np.isfinite(probe).all()
assert probe_norms.min() > 0.995
assert probe_norms.max() < 1.005

print("SHARD ROW LOADER — PASS")


## 7. Frozen metrics and feedback implementation


In [ ]:
def normalize_rows(x):
    x = np.asarray(x, np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, 1e-12)

def evaluate_batch(qids, ranked_ids, k=10):
    recall = np.empty(len(qids), np.float32)
    mrr = np.empty(len(qids), np.float32)
    ndcg = np.empty(len(qids), np.float32)
    discounts = 1.0 / np.log2(np.arange(2, k + 2))

    for i, qid in enumerate(qids):
        relset = dev_qrels.get(str(qid), set())
        ranked = ranked_ids[i, :k]

        hits = np.asarray(
            [1.0 if int(d) in relset else 0.0 for d in ranked],
            dtype=np.float32,
        )

        recall[i] = hits.sum() / max(len(relset), 1)
        pos = np.flatnonzero(hits)
        mrr[i] = 1.0 / (pos[0] + 1) if len(pos) else 0.0

        dcg = float((hits * discounts).sum())
        ideal = min(len(relset), k)
        idcg = float(discounts[:ideal].sum()) if ideal else 0.0
        ndcg[i] = dcg / idcg if idcg > 0 else 0.0

    return recall, mrr, ndcg

def jaccard_rows(a, b):
    vals = np.empty(len(a), np.float32)
    for i in range(len(a)):
        A = set(map(int, a[i]))
        B = set(map(int, b[i]))
        vals[i] = len(A & B) / max(len(A | B), 1)
    return vals

def cosine_distance_rows(a, b):
    a = normalize_rows(a)
    b = normalize_rows(b)
    return 1.0 - np.sum(a * b, axis=1)

def feedback_matrix(ids, scores, config):
    k = int(config["k"])
    dids = np.asarray(ids[:, :k], dtype=np.int64)
    x = load_rows_from_shards_fast(dids)

    if config["method"] == "mean":
        f = x.mean(axis=1)

    elif config["method"] == "softmax":
        temp = float(config["temperature"])
        z = np.asarray(scores[:, :k], np.float64) / temp
        z -= z.max(axis=1, keepdims=True)

        w = np.exp(np.clip(z, -60, 60))
        w /= np.maximum(w.sum(axis=1, keepdims=True), 1e-12)

        f = (x * w[:, :, None]).sum(axis=1)

    else:
        raise ValueError(config["method"])

    f = f.astype(np.float32)
    f /= np.maximum(np.linalg.norm(f, axis=1, keepdims=True), 1e-12)
    return f

def anchored_update_matrix(q0, feedback, alpha):
    q = ((1.0 - float(alpha)) * q0 + float(alpha) * feedback).astype(np.float32)
    q /= np.maximum(np.linalg.norm(q, axis=1, keepdims=True), 1e-12)
    return q

def config_key(config):
    temp = "none" if config.get("temperature") is None else str(config["temperature"]).replace(".", "p")
    return (
        f"{config['method']}"
        f"-k{int(config['k'])}"
        f"-a{str(config['alpha']).replace('.', 'p')}"
        f"-t{temp}"
    )


## 8. Synchronized PQ32 and SQ8 self-feedback trajectories


In [ ]:
DEV_Q0 = normalize_rows(
    np.asarray(queries[dev_query_rows], dtype=np.float32)
)

def run_self_feedback(index_path, condition, config):
    index = faiss.read_index(str(index_path))
    index.nprobe = NPROBE

    q0 = DEV_Q0
    qt = q0.copy()
    states = []

    for t in range(MAX_ROUNDS + 1):
        print(condition, config_key(config), "iteration", t)
        t0 = time.time()

        scores, ids = index.search(
            np.ascontiguousarray(qt, np.float32),
            TOP_RETRIEVE,
        )

        recall, mrr, ndcg = evaluate_batch(dev_ids, ids, TOP_K)

        states.append({
            "q": qt.copy(),
            "ids": ids.copy(),
            "scores": scores.copy(),
            "recall": recall,
            "mrr": mrr,
            "ndcg": ndcg,
        })

        print(
            "  R@10=", float(recall.mean()),
            "nDCG@10=", float(ndcg.mean()),
            "elapsed=", round(time.time() - t0, 1), "s"
        )

        if t < MAX_ROUNDS:
            fb = feedback_matrix(ids, scores, config)
            qt = anchored_update_matrix(q0, fb, config["alpha"])

    del index
    gc.collect()
    return states

state_cache = {}

for config in FROZEN_FEEDBACK_CONFIGS:
    ck = config_key(config)

    for condition, path in [
        ("ivfpq32", PQ32_PATH),
        ("ivfsq8", SQ8_PATH),
    ]:
        cache_path = OUT / f"states-{condition}-{ck}.npz"

        states = run_self_feedback(path, condition, config)

        payload = {}
        for t, s in enumerate(states):
            for field in ["q", "ids", "scores", "recall", "mrr", "ndcg"]:
                payload[f"{field}_{t}"] = s[field]

        np.savez_compressed(cache_path, **payload)
        print("Saved:", cache_path)

        state_cache[(condition, ck)] = states

print("SYNCHRONIZED TRAJECTORIES — COMPLETE")


## 9. H1–H3 query-level endpoints


In [ ]:
trajectory_rows = []

for config in FROZEN_FEEDBACK_CONFIGS:
    ck = config_key(config)
    A = state_cache[("ivfpq32", ck)]
    B = state_cache[("ivfsq8", ck)]

    candidate_t0 = None

    for t in range(MAX_ROUNDS + 1):
        dq = cosine_distance_rows(A[t]["q"], B[t]["q"])
        dc = 1.0 - jaccard_rows(A[t]["ids"], B[t]["ids"])

        if t == 0:
            candidate_t0 = dc.copy()

        dc_inc = dc - candidate_t0
        ugap = B[t]["ndcg"] - A[t]["ndcg"]
        augap = np.abs(ugap)

        for i, qid in enumerate(dev_ids):
            trajectory_rows.append({
                "query_id": qid,
                "method": config["method"],
                "config_key": ck,
                "iteration": t,
                "query_divergence": float(dq[i]),
                "candidate_divergence": float(dc[i]),
                "candidate_divergence_increment": float(dc_inc[i]),
                "utility_gap": float(ugap[i]),
                "abs_utility_gap": float(augap[i]),
                "pq32_ndcg": float(A[t]["ndcg"][i]),
                "sq8_ndcg": float(B[t]["ndcg"][i]),
            })

trajectory_df = pd.DataFrame(trajectory_rows)
trajectory_df.to_parquet(
    OUT / "sealed_hotpotqa_dev_paired_trajectories.parquet",
    index=False,
)

def linear_slope(g, metric):
    g = g.sort_values("iteration")
    x = g["iteration"].to_numpy(np.float64)
    y = g[metric].to_numpy(np.float64)
    return float(np.polyfit(x, y, 1)[0])

method_slope_rows = []

for (qid, method, ck), g in trajectory_df.groupby(
    ["query_id", "method", "config_key"]
):
    method_slope_rows.append({
        "query_id": qid,
        "method": method,
        "config_key": ck,
        "H1_query_divergence_slope": linear_slope(g, "query_divergence"),
        "H2_candidate_increment_slope": linear_slope(g, "candidate_divergence_increment"),
        "H3_abs_utility_gap_slope": linear_slope(g, "abs_utility_gap"),
    })

method_slopes = pd.DataFrame(method_slope_rows)

query_slopes = (
    method_slopes
    .groupby("query_id", as_index=False)[[
        "H1_query_divergence_slope",
        "H2_candidate_increment_slope",
        "H3_abs_utility_gap_slope",
    ]]
    .mean()
)

display(query_slopes.describe())
print("H1-H3 ENDPOINTS — READY")


## 10. H4 frozen feedback-source intervention


In [ ]:
def run_intervention(config):
    pq32 = faiss.read_index(str(PQ32_PATH))
    sq8 = faiss.read_index(str(SQ8_PATH))

    pq32.nprobe = NPROBE
    sq8.nprobe = NPROBE

    q0 = DEV_Q0
    qA = q0.copy()
    qB = q0.copy()
    rows = []

    for t in range(MAX_ROUNDS + 1):
        print("intervention", config_key(config), "iteration", t)

        sA, idA = pq32.search(
            np.ascontiguousarray(qA, np.float32),
            TOP_RETRIEVE,
        )

        # B utility remains evaluated by PQ32 search.
        sB_search, idB_search = pq32.search(
            np.ascontiguousarray(qB, np.float32),
            TOP_RETRIEVE,
        )

        # Only B feedback source changes to SQ8.
        sB_fb, idB_fb = sq8.search(
            np.ascontiguousarray(qB, np.float32),
            TOP_RETRIEVE,
        )

        rA, mA, nA = evaluate_batch(dev_ids, idA, TOP_K)
        rB, mB, nB = evaluate_batch(dev_ids, idB_search, TOP_K)

        for i, qid in enumerate(dev_ids):
            rows.append({
                "query_id": qid,
                "method": config["method"],
                "config_key": config_key(config),
                "iteration": t,
                "A_ndcg": float(nA[i]),
                "B_ndcg": float(nB[i]),
            })

        if t < MAX_ROUNDS:
            fA = feedback_matrix(idA, sA, config)
            fB = feedback_matrix(idB_fb, sB_fb, config)

            qA = anchored_update_matrix(q0, fA, config["alpha"])
            qB = anchored_update_matrix(q0, fB, config["alpha"])

    del pq32, sq8
    gc.collect()

    return pd.DataFrame(rows)

intervention_frames = []

for config in FROZEN_FEEDBACK_CONFIGS:
    ck = config_key(config)
    cache_path = OUT / f"intervention-{ck}.parquet"

    df = run_intervention(config)
    df.to_parquet(cache_path, index=False)
    print("Saved:", cache_path)

    intervention_frames.append(df)

intervention_df = pd.concat(intervention_frames, ignore_index=True)

final = intervention_df[
    intervention_df["iteration"] == MAX_ROUNDS
].copy()

final["H4_B_minus_A"] = final["B_ndcg"] - final["A_ndcg"]

method_intervention = final[
    ["query_id", "method", "config_key", "H4_B_minus_A"]
].copy()

query_intervention = (
    method_intervention
    .groupby("query_id", as_index=False)["H4_B_minus_A"]
    .mean()
)

display(query_intervention.describe())
print("H4 ENDPOINT — READY")


## 11. Query-level bootstrap, sign-flip, Holm


In [ ]:
def bootstrap_mean_chunked(values, samples=BOOTSTRAP_SAMPLES, seed=0, chunk=RESAMPLE_CHUNK):
    x = np.asarray(values, np.float64)
    x = x[np.isfinite(x)]
    n = len(x)

    rng = np.random.default_rng(seed)
    draws = []
    remaining = samples

    while remaining > 0:
        b = min(chunk, remaining)
        idx = rng.integers(0, n, size=(b, n), dtype=np.int32)
        draws.append(x[idx].mean(axis=1))
        remaining -= b

    draws = np.concatenate(draws)

    return {
        "n": int(n),
        "mean": float(x.mean()),
        "median": float(np.median(x)),
        "ci_low": float(np.quantile(draws, 0.025)),
        "ci_high": float(np.quantile(draws, 0.975)),
    }

def sign_flip_chunked(values, samples=RANDOMIZATION_SAMPLES, seed=0, chunk=RESAMPLE_CHUNK):
    x = np.asarray(values, np.float64)
    x = x[np.isfinite(x)]

    n = len(x)
    obs = float(x.mean())
    rng = np.random.default_rng(seed)

    exceed = 0
    done = 0

    while done < samples:
        b = min(chunk, samples - done)
        signs = rng.integers(0, 2, size=(b, n), dtype=np.int8)
        signs = signs.astype(np.float32) * 2.0 - 1.0
        stats = (signs * x[None, :]).mean(axis=1)

        exceed += int(np.sum(stats >= obs))
        done += b

    return (exceed + 1) / (samples + 1)

def holm_adjust(pvalues):
    p = np.asarray(pvalues, np.float64)
    m = len(p)

    order = np.argsort(p)
    adjusted = np.empty(m, np.float64)
    running = 0.0

    for rank, idx in enumerate(order):
        raw = (m - rank) * p[idx]
        running = max(running, raw)
        adjusted[idx] = min(running, 1.0)

    return adjusted

endpoint_values = {
    "H1_query_divergence_slope":
        query_slopes["H1_query_divergence_slope"].to_numpy(np.float64),

    "H2_candidate_increment_slope":
        query_slopes["H2_candidate_increment_slope"].to_numpy(np.float64),

    "H3_abs_utility_gap_slope":
        query_slopes["H3_abs_utility_gap_slope"].to_numpy(np.float64),

    "H4_B_minus_A":
        query_intervention["H4_B_minus_A"].to_numpy(np.float64),
}

rows = []

for i, (name, x) in enumerate(endpoint_values.items()):
    boot = bootstrap_mean_chunked(x, seed=SEED + 1000 + i)
    p = sign_flip_chunked(x, seed=SEED + 2000 + i)

    rows.append({
        "endpoint": name,
        **boot,
        "randomization_p": p,
        "fraction_positive": float(np.mean(x > 0)),
        "fraction_zero": float(np.mean(np.isclose(x, 0, atol=1e-12))),
        "fraction_negative": float(np.mean(x < 0)),
    })

confirm_df = pd.DataFrame(rows)
confirm_df["holm_p"] = holm_adjust(confirm_df["randomization_p"])

confirm_df["pass"] = (
    (confirm_df["mean"] > 0)
    & (confirm_df["ci_low"] > 0)
    & (confirm_df["holm_p"] < 0.05)
)

display(confirm_df)


## 12. Secondary robustness and FEVER effect preservation


In [ ]:
secondary = []

for method, g in method_slopes.groupby("method"):
    for metric in [
        "H1_query_divergence_slope",
        "H2_candidate_increment_slope",
        "H3_abs_utility_gap_slope",
    ]:
        x = (
            g.groupby("query_id")[metric]
            .mean()
            .to_numpy(np.float64)
        )

        b = bootstrap_mean_chunked(
            x,
            samples=10_000,
            seed=SEED + 3000,
        )

        secondary.append({
            "method": method,
            "endpoint": metric,
            **b,
            "fraction_positive": float(np.mean(x > 0)),
        })

for method, g in method_intervention.groupby("method"):
    x = (
        g.groupby("query_id")["H4_B_minus_A"]
        .mean()
        .to_numpy(np.float64)
    )

    b = bootstrap_mean_chunked(
        x,
        samples=10_000,
        seed=SEED + 4000,
    )

    secondary.append({
        "method": method,
        "endpoint": "H4_B_minus_A",
        **b,
        "fraction_positive": float(np.mean(x > 0)),
    })

secondary_df = pd.DataFrame(secondary)

FEVER_DEV_REFERENCE = {
    "H1_query_divergence_slope": 0.00410901625977956,
    "H2_candidate_increment_slope": 0.008156594857148784,
    "H3_abs_utility_gap_slope": 0.0072613631825287185,
    "H4_B_minus_A": 0.030375916320252733,
}

preservation = []

for _, r in confirm_df.iterrows():
    ref = FEVER_DEV_REFERENCE[r["endpoint"]]

    preservation.append({
        "endpoint": r["endpoint"],
        "FEVER_DEV_reference": ref,
        "HotpotQA_DEV_effect": float(r["mean"]),
        "HotpotQA_over_FEVER": float(r["mean"] / ref),
        "same_direction": bool(np.sign(r["mean"]) == np.sign(ref)),
    })

preservation_df = pd.DataFrame(preservation)

display(secondary_df)
display(preservation_df)


## 13. Final sealed decision


In [ ]:
all_pass = bool(confirm_df["pass"].all())

print("=== ARC-v0.8 SEALED HOTPOTQA CROSS-DATASET CONFIRMATION ===")
print("Protocol SHA-256:", protocol_sha256)
print()

display(
    confirm_df[[
        "endpoint",
        "mean",
        "ci_low",
        "ci_high",
        "randomization_p",
        "holm_p",
        "fraction_positive",
        "fraction_zero",
        "fraction_negative",
        "pass",
    ]]
)

print()
display(preservation_df)

if all_pass:
    decision = (
        "SEALED CROSS-DATASET CONFIRMATION PASS: all four FEVER-preregistered "
        "approximation-feedback endpoints replicate on HotpotQA DEV under "
        "query-level inference and joint Holm family-wise correction."
    )
else:
    failed = confirm_df.loc[~confirm_df["pass"], "endpoint"].tolist()

    decision = (
        "SEALED CROSS-DATASET CONFIRMATION FAIL/PARTIAL: failed endpoints: "
        + ", ".join(failed)
        + ". Do not claim complete cross-dataset replication; any subsequent "
        "diagnosis must be explicitly labeled post-confirmatory."
    )

print("DECISION:", decision)


## 14. Save evidence and final report hash


In [ ]:
query_slopes.to_csv(
    OUT / "hotpotqa_dev_query_pooled_slopes.csv",
    index=False,
)

method_slopes.to_csv(
    OUT / "hotpotqa_dev_query_method_slopes.csv",
    index=False,
)

query_intervention.to_csv(
    OUT / "hotpotqa_dev_query_pooled_intervention.csv",
    index=False,
)

method_intervention.to_csv(
    OUT / "hotpotqa_dev_query_method_intervention.csv",
    index=False,
)

confirm_df.to_csv(
    OUT / "confirmatory_endpoints.csv",
    index=False,
)

secondary_df.to_csv(
    OUT / "secondary_method_robustness.csv",
    index=False,
)

preservation_df.to_csv(
    OUT / "cross_dataset_effect_preservation.csv",
    index=False,
)

report = {
    "status": "SEALED_HOTPOTQA_H1_H4_CONFIRMATION_COMPLETE",
    "protocol_sha256": protocol_sha256,
    "source_protocol": str(PROTOCOL_PATH),
    "hotpotqa_dev_query_count": len(dev_ids),
    "feedback_configs": FROZEN_FEEDBACK_CONFIGS,
    "primary_endpoints": confirm_df.to_dict(orient="records"),
    "cross_dataset_effect_preservation": preservation_df.to_dict(orient="records"),
    "secondary_method_robustness": secondary_df.to_dict(orient="records"),
    "all_primary_endpoints_pass": all_pass,
    "decision": decision,
    "hotpotqa_h1_h4_based_selection_performed": False,
    "hotpotqa_baseline_was_observed_for_infrastructure_validation": True,
    "test_retrieval_performed": False,
    "test_qrels_accessed": False,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = OUT / "report.json"

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
        default=float,
    ),
    encoding="utf-8",
)

report_sha = hashlib.sha256(REPORT_PATH.read_bytes()).hexdigest()

(OUT / "FINAL_REPORT_SHA256.txt").write_text(
    report_sha + "\n",
    encoding="utf-8",
)

print("Saved:", OUT)
print("Report:", REPORT_PATH)
print("Report SHA-256:", report_sha)


## Interpretation rule

The primary result is the four-endpoint sealed decision.

- If all four pass: report an independent HotpotQA cross-dataset confirmation under the frozen FEVER design.
- If any endpoint fails: report partial/non-replication at the endpoint level.
- Do not change feedback parameters, endpoint definitions, pooling, or statistical tests after observing H1–H4 and then call the changed analysis confirmatory.
- Any later PQ64 dose-response, subgroup analysis, alternative feedback policy, or mitigation experiment must be explicitly labeled secondary/post-confirmatory.
